In [45]:
import pandas as pd

In [46]:
df=pd.read_csv('resume_extracted.csv')
selected_categories = [
    "INFORMATION-TECHNOLOGY",
    "BUSINESS-DEVELOPMENT",
    "ACCOUNTANT",
    "ENGINEERING",
    "FINANCE",
    "BANKING",
    "HEALTHCARE",
    "DESIGNER",
    "TEACHER",
    "HR"
]
df = df[df["category"].isin(selected_categories)]
df.dropna(subset=['resume_text','category'],inplace=True)
df = df.reset_index(drop=True)
df=df.drop_duplicates(subset=['resume_text'])
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words=set(stopwords.words('english'))
keep_words = {'no', 'nor', 'not', 'without', 'against', 'only', 'but'}
stop_words=stop_words-keep_words
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')
import re
import html
def data_preprocessing(text):
    text = html.unescape(text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s+#./-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'\b\d{10,}\b', ' ', text)
    text = re.sub(r'\.(?=\s|$)', ' ', text)
    text=text.lower()
    if len(text)<50:
        return ""
    tokens=text.split()
    new_tokens=[]
    for word in tokens:
        if word not in stop_words:
            new_tokens.append(word)
    tokens=new_tokens

    lemmatizer = WordNetLemmatizer()

    def get_wordnet_pos(tag):
        if tag.startswith('J'):
            return wordnet.ADJ
        elif tag.startswith('V'):
            return wordnet.VERB
        elif tag.startswith('N'):
            return wordnet.NOUN
        elif tag.startswith('R'):
            return wordnet.ADV
        return wordnet.NOUN

    pos_tags = nltk.pos_tag(tokens)

    lemmatized_tokens = []

    for word, tag in pos_tags:
        pos = get_wordnet_pos(tag)
        lemmatized_word = lemmatizer.lemmatize(word, pos)
        lemmatized_tokens.append(lemmatized_word)

    tokens = lemmatized_tokens 
    return ' '.join(tokens)
from sklearn.preprocessing import FunctionTransformer
preprocessor=FunctionTransformer(
    lambda X:X.apply(data_preprocessing),
    validate=False
)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hites\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\hites\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hites\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hites\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [47]:
df.head()

,filename,resume_text,category
0,10554236.pdf,ACCOUNTANT\nSummary\nFinancial Accountant spec...,ACCOUNTANT
1,10674770.pdf,STAFF ACCOUNTANT\nSummary\nHighly analytical a...,ACCOUNTANT
2,11163645.pdf,ACCOUNTANT\nProfessional Summary\nTo obtain a ...,ACCOUNTANT
3,11759079.pdf,SENIOR ACCOUNTANT\nExperience\nCompany Name Ju...,ACCOUNTANT
4,12065211.pdf,SENIOR ACCOUNTANT\nProfessional Summary\nSenio...,ACCOUNTANT


In [48]:
df['category'].value_counts()

category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      119
ACCOUNTANT                118
ENGINEERING               118
FINANCE                   117
BANKING                   115
HEALTHCARE                115
HR                        110
DESIGNER                  107
TEACHER                   102
Name: count, dtype: int64

In [50]:
from sklearn.pipeline import Pipeline
from scipy.stats import randint
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
model=Pipeline([('preprocessing',preprocessor),('tfidf',TfidfVectorizer(max_features=20000,ngram_range=(1,2),sublinear_tf=True)),('model',LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=42
    ))])
X=df['resume_text']
y=df['category']
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


In [ ]:
# param_dist = {
#     "model__n_estimators": [100, 200, 300],
#     "model__max_depth": [None, 20, 40, 60],
#     "model__min_samples_split": randint(2, 11),
#     "model__min_samples_leaf": randint(1, 5),
#     "model__max_features": ["sqrt", 0.3, 0.5]
# }

# # Randomized search
# random_search = RandomizedSearchCV(
#     estimator=model,
#     param_distributions=param_dist,
#     n_iter=5,
#     cv=2,
#     scoring="f1_macro",
#     random_state=42,
#     n_jobs=1,
#     verbose=2,
#     error_score="raise"
# )

In [53]:
param_grid = {
    'tfidf__max_features': [10000, 20000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'model__C': [0.1, 1, 10]
}

# Grid search
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=1,
    verbose=2
)

In [54]:
grid_search.fit(X_train,y_train)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END model__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 1); total time=  22.3s
[CV] END model__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 1); total time=  21.9s
[CV] END model__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 1); total time=  21.9s
[CV] END model__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 2); total time=  22.9s
[CV] END model__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 2); total time=  22.7s
[CV] END model__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 2); total time=  22.8s
[CV] END model__C=0.1, tfidf__max_features=20000, tfidf__ngram_range=(1, 1); total time=  21.8s
[CV] END model__C=0.1, tfidf__max_features=20000, tfidf__ngram_range=(1, 1); total time=  21.8s
[CV] END model__C=0.1, tfidf__max_features=20000, tfidf__ngram_range=(1, 1); total time=  21.8s
[CV] END model__C=0.1, tfidf__max_features=20000, tfidf__ngram_range=(1, 2)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.1, 1, ...], 'tfidf__max_features': [10000, 20000], 'tfidf__ngram_range': [(1, ...), (1, ...)]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model

In [ ]:
# print("best parameters are:",random_search.best_params_)

best parameters are: {'model__max_depth': 20, 'model__max_features': 0.3, 'model__min_samples_leaf': 4, 'model__min_samples_split': 6, 'model__n_estimators': 100}


In [3]:
y_pred=grid_search.predict(X_test)

NameError: name 'grid_search' is not defined

In [56]:
from sklearn.metrics import accuracy_score,precision_score,recall_score
print(accuracy_score(y_test,y_pred))
print("Precision:", precision_score(
    y_test, y_pred, average='weighted', zero_division=0
))

print("Recall:", recall_score(
    y_test, y_pred, average='weighted', zero_division=0
))

0.8777292576419214
Precision: 0.8821182461544996
Recall: 0.8777292576419214


In [57]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, zero_division=0))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.75      0.88      0.81        24
               BANKING       0.90      0.83      0.86        23
  BUSINESS-DEVELOPMENT       0.85      0.92      0.88        24
              DESIGNER       0.91      1.00      0.95        21
           ENGINEERING       0.95      0.88      0.91        24
               FINANCE       0.85      0.71      0.77        24
            HEALTHCARE       0.95      0.78      0.86        23
                    HR       0.85      1.00      0.92        22
INFORMATION-TECHNOLOGY       0.88      0.88      0.88        24
               TEACHER       0.95      0.95      0.95        20

              accuracy                           0.88       229
             macro avg       0.88      0.88      0.88       229
          weighted avg       0.88      0.88      0.88       229



In [1]:
resume_text = """
Dedicated mathematics teacher with a B.Ed. and M.Sc. in Mathematics.
Experienced in classroom management, lesson planning, curriculum
development, student assessment, and academic support.
Skilled in educational technology, smart boards, and online teaching.
"""

In [2]:
model.predict(resume_text)

NameError: name 'model' is not defined